In [0]:
%pip install tqdm

In [0]:
import os
import pandas as pd
from tqdm import tqdm

In [0]:
# ── Unity Catalog location (must match the other scripts in this pipeline) ──
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #Change
SCHEMA = "tier1_raw" #Change
 
# Source table: the flight inventory table generated by Job 1 (1_Flight_table).
flight_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"
# Destination table: where this script saves the plot-clip inventory.
TABLE_NAME_CLIP = f"{CATALOG}.{SCHEMA}.drone_plot_clipped_table"

In [0]:
# Load the full flight inventory into a local pandas DataFrame so we can
# loop through it row by row.
flight_df = spark.read.table(flight_table).toPandas()
len(flight_df)

In [0]:
row_list = []
 
# ── Go through every flight and check its plot-clip status ─────────────────
for idx, row in tqdm(flight_df.iterrows(), total=len(flight_df)):
 
    # 1. We obtain the base route
    flight_path = os.path.dirname(row['flight_metadata_path'])
 
    # DATABRICKS FIX: make sure 'os' can read the path correctly by converting
    # the "dbfs:/" prefix to "/dbfs/", which is what standard Python file
    # operations expect.
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
 
    # 2. We point to the folder of clipped plots (lowercase by convention).
    # This is where Job 3's plot-clipping step (3_orquestator / pending_clips_gen)
    # is expected to save each flight's clipped plot images.
    plot_clipped_path = f"{flight_path}/plot_clipped"
    plot_count = 0
 
    # 3. We count the images REGARDLESS of whether the folder exists yet or not.
    # If the folder doesn't exist, plot_count simply stays at 0 — this flight
    # just hasn't had its plots clipped yet.
    if os.path.exists(plot_clipped_path):
        files = os.listdir(plot_clipped_path)
        plot_count = len([f for f in files if f.lower().endswith(('.tif', '.jpg', '.jpeg', '.png'))])
 
    # 4. We ALWAYS build the dictionary to maintain a complete inventory
    # (every flight gets a row, whether it has clipped plots yet or not —
    # this is what lets the orchestrator later identify pending flights).
    row_dict = {
        'site': row['site'],
        'trial': row['trial'],
        'season': row['season'],
        'field':  row['field'],
        'location': row['location'],
        'mission': row['mission'],
        'flight_date': row['flight_date'],
 
        # Specific routes and metrics for this stage
        'flight_metadata_path': row['flight_metadata_path'],
        'plot_clipped_path': plot_clipped_path,
        'plots_exist': plot_count > 0, # Flag equivalent to 'ortho_exists'
        'plot_image_count': plot_count
    }
 
    row_list.append(row_dict)
 
# 5. We generate the final DataFrame
plot_df = pd.DataFrame(row_list)
 
print(f"Total flights processed for plots inventory: {len(plot_df)}")
 
# 6. Permanent saving in Delta Table (adapted from your previous version)
if len(plot_df) > 0:
    display(plot_df)
else:
    # If this is empty, it usually means Job 1 (flight table generation)
    # hasn't been run yet, since this script depends on that table existing.
    print("It's still empty. Check if you ran the flight_df table before running this.")

In [0]:
# Convert to a Spark DataFrame so it can be saved as a table in the catalog.
spark_df = spark.createDataFrame(plot_df)
 
spark_df.printSchema()

In [0]:
# Write (overwrite) the plot-clip inventory table. "mergeSchema" allows the
# table's schema to evolve if new columns are added in future runs.
spark_df.write.option("mergeSchema", "true").saveAsTable(TABLE_NAME_CLIP, mode="overwrite")